In [53]:
import yfinance as yf
import pandas as pd
import statsmodels.api as sm

## AWS

In [54]:
# AWS peers and benchmark
tickers_aws = [
    "MSFT", "ORCL", "NTNX", "OVH.PA",
    "^SP500TR"
]

fx_tickers_aws = ["EURUSD=X"]


# Download monthly adjusted prices
prices_aws = yf.download(
    tickers_aws,
    start="2021-10-01",
    end="2026-11-01",
    interval="1mo",
    auto_adjust=True,
    progress=False
)["Close"]

# Download monthly exchange rates
fx_aws = yf.download(
    fx_tickers_aws,
    start="2021-10-01",
    end="2026-11-01",
    interval="1mo",
    auto_adjust=False,
    progress=False
)["Close"]

display(prices_aws.tail())
display(fx_aws.tail())

Ticker,MSFT,NTNX,ORCL,OVH.PA,^SP500TR
Date,,,,,
2026-06-01,372.319092,50.959999,145.503784,14.38,16774.070312
2026-07-01,463.846802,59.009998,129.391434,14.94,16763.419922
2026-08-01,507.290009,68.570000,148.570511,14.96,17219.939453
2026-09-01,512.900024,70.410004,136.794067,16.58,17160.410156
2026-10-01,521.909973,67.209999,130.669998,13.25,17011.140625


Ticker,EURUSD=X
Date,
2026-06-01,1.142204
2026-07-01,1.152353
2026-08-01,1.158896
2026-09-01,1.134083
2026-10-01,1.120699


In [55]:
# Convert share prices into USD
prices_usd_aws = pd.DataFrame(index=prices_aws.index)

prices_usd_aws["MSFT"] = prices_aws["MSFT"]
prices_usd_aws["ORCL"] = prices_aws["ORCL"]
prices_usd_aws["NTNX"] = prices_aws["NTNX"]

prices_usd_aws["OVH.PA"] = (
    prices_aws["OVH.PA"] * fx_aws["EURUSD=X"]
)

prices_usd_aws["^SP500TR"] = prices_aws["^SP500TR"]

display(prices_usd_aws.tail())


,MSFT,ORCL,NTNX,OVH.PA,^SP500TR
Date,,,,,
2026-06-01,372.319092,145.503784,50.959999,16.424899,16774.070312
2026-07-01,463.846802,129.391434,59.009998,17.216147,16763.419922
2026-08-01,507.290009,148.570511,68.570000,17.337087,17219.939453
2026-09-01,512.900024,136.794067,70.410004,18.803089,17160.410156
2026-10-01,521.909973,130.669998,67.209999,14.849266,17011.140625


In [56]:
# 1. Check dataset shape
print("Dataset shape:", prices_usd_aws.shape)

# 2. Check date range
print("\nFirst date:", prices_usd_aws.index.min())
print("Last date:", prices_usd_aws.index.max())

# 3. Check missing values
print("\nMissing values per ticker:")
print(prices_usd_aws.isna().sum())

# 4. Check duplicate dates
print("\nDuplicate dates:", prices_usd_aws.index.duplicated().sum())

# 5. Check chronological order
print("\nDates sorted:", prices_usd_aws.index.is_monotonic_increasing)

# 6. Check non-positive prices
print("\nNon-positive prices:")
print((prices_usd_aws <= 0).sum())

# 7. Check data types
print("\nData types:")
print(prices_usd_aws.dtypes)


Dataset shape: (61, 5)

First date: 2021-10-01 00:00:00
Last date: 2026-10-01 00:00:00

Missing values per ticker:
MSFT        0
ORCL        0
NTNX        0
OVH.PA      0
^SP500TR    0
dtype: int64

Duplicate dates: 0

Dates sorted: True

Non-positive prices:
MSFT        0
ORCL        0
NTNX        0
OVH.PA      0
^SP500TR    0
dtype: int64

Data types:
MSFT        float64
ORCL        float64
NTNX        float64
OVH.PA      float64
^SP500TR    float64
dtype: object


In [57]:
# Calculate monthly returns
returns_aws = prices_usd_aws.pct_change(
    fill_method=None
).iloc[1:]

display(returns_aws.head())

print("Number of observations:", len(returns_aws))
print("\nMissing values:")
print(returns_aws.isna().sum())

,MSFT,ORCL,NTNX,OVH.PA,^SP500TR
Date,,,,,
2021-11-01,-0.001282,-0.054200,-0.031769,-0.015664,-0.006929
2021-12-01,0.017332,-0.038902,-0.040939,0.253805,0.044816
2022-01-01,-0.075345,-0.065915,-0.141871,-0.081154,-0.051747
2022-02-01,-0.037212,-0.063948,-0.023409,-0.049672,-0.029942
2022-03-01,0.031862,0.088982,0.004494,0.036719,0.037130


Number of observations: 60

Missing values:
MSFT        0
ORCL        0
NTNX        0
OVH.PA      0
^SP500TR    0
dtype: int64


In [58]:
for ticker in ["MSFT", "NTNX", "ORCL", "^SP500TR"]:
    print(f"\n{ticker}")
    print("Largest gain:", returns_aws[ticker].idxmax(),
          f"{returns_aws[ticker].max():.2%}")
    print("Largest loss:", returns_aws[ticker].idxmin(),
          f"{returns_aws[ticker].min():.2%}")


# Inspect extreme-return months alongside market returns
for ticker in ["MSFT", "NTNX", "ORCL"]:
    max_month = returns_aws[ticker].idxmax()
    min_month = returns_aws[ticker].idxmin()

    print(f"\n{ticker}")
    for month in [max_month, min_month]:
        print(
            month.strftime("%Y-%m"),
            f"Stock: {returns_aws.loc[month, ticker]:.2%}",
            f"Market: {returns_aws.loc[month, '^SP500TR']:.2%}"
        )


MSFT
Largest gain: 2026-07-01 00:00:00 24.58%
Largest loss: 2026-06-01 00:00:00 -17.15%

NTNX
Largest gain: 2022-10-01 00:00:00 31.54%
Largest loss: 2022-05-01 00:00:00 -35.28%

ORCL
Largest gain: 2026-05-01 00:00:00 39.90%
Largest loss: 2026-06-01 00:00:00 -35.09%

^SP500TR
Largest gain: 2026-04-01 00:00:00 10.49%
Largest loss: 2022-09-01 00:00:00 -9.21%

MSFT
2026-07 Stock: 24.58% Market: -0.06%
2026-06 Stock: -17.15% Market: -0.95%

NTNX
2022-10 Stock: 31.54% Market: 8.10%
2022-05 Stock: -35.28% Market: 0.18%

ORCL
2026-05 Stock: 39.90% Market: 5.26%
2026-06 Stock: -35.09% Market: -0.95%


### OLS regression

In [59]:
# Run OLS regressions against S&P 500 Total Return Index
aws_results = []

for ticker in ["MSFT", "ORCL", "NTNX", "OVH.PA"]:
    data = returns_aws[[ticker, "^SP500TR"]].dropna()

    X = sm.add_constant(data["^SP500TR"])
    y = data[ticker]

    model = sm.OLS(y, X).fit()

    aws_results.append({
        "Ticker": ticker,
        "Equity Beta": model.params["^SP500TR"],
        "Alpha": model.params["const"],
        "R-squared": model.rsquared,
        "Beta p-value": model.pvalues["^SP500TR"],
        "Observations": int(model.nobs)
    })

beta_results = pd.DataFrame(aws_results).set_index("Ticker")

display(beta_results.round(4))


,Equity Beta,Alpha,R-squared,Beta p-value,Observations
Ticker,,,,,
MSFT,1.0473,0.0000,0.3783,0.0000,60
ORCL,1.7900,-0.0039,0.3551,0.0000,60
NTNX,0.7215,0.0122,0.0620,0.0550,60
OVH.PA,1.2615,-0.0045,0.0903,0.0197,60


In [60]:
# Function to calculate betas

def calculate_peer_beta(
    ticker,
    share_price,
    shares_outstanding_bn,
    current_debt_bn,
    long_term_debt_bn,
    cash_bn,
    short_term_investments_bn=0
):
    """
    Calculate a peer's market capitalisation, unlevered beta,
    and cash-corrected unlevered beta.

    Monetary inputs are in USD billions.
    Share price is in USD per share.
    Equity beta is taken from the existing OLS regression results.

    Assumptions:
    - Zero debt beta
    - No tax adjustment
    - Cash and short-term investments have zero beta
    - Operating lease liabilities excluded from financial debt
    """

    # Retrieve the equity beta estimated by OLS
    equity_beta = beta_results.loc[ticker, "Equity Beta"]

    # Market capitalisation
    market_cap = share_price * shares_outstanding_bn

    # Total financial debt
    total_debt = current_debt_bn + long_term_debt_bn

    # Cash and cash equivalents + short-term investments
    total_cash = cash_bn + short_term_investments_bn

    # Unlever the equity beta
    unlevered_beta = equity_beta / (1 + total_debt / market_cap)

    # Correct the unlevered beta for cash
    firm_value = market_cap + total_debt

    cash_corrected_beta = unlevered_beta / (
        1 - total_cash / firm_value
    )

    return {
        "Ticker": ticker,
        "Equity Beta": equity_beta,
        "Market Cap (bn)": market_cap,
        "Debt (bn)": total_debt,
        "Cash + Investments (bn)": total_cash,
        "Unlevered Beta": unlevered_beta,
        "Cash-Corrected Beta": cash_corrected_beta
    }


### Microsoft

In [61]:
# Microsoft's closing share price on 30 October 2026
msft_price = yf.Ticker("MSFT").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

msft_share_price = float(msft_price["Close"].iloc[0])
print(f"Microsoft share price: ${msft_share_price:.2f}")

Microsoft share price: $521.91


In [62]:
msft_result = calculate_peer_beta(
    ticker="MSFT",
    share_price=msft_share_price,
    shares_outstanding_bn=7.427,
    current_debt_bn=4.391,
    long_term_debt_bn=31.090,
    cash_bn=20.281,
    short_term_investments_bn=44.228
)

display(pd.DataFrame([msft_result]).set_index("Ticker").T.round(4))

Ticker,MSFT
Equity Beta,1.0473
Market Cap (bn),3876.2254
Debt (bn),35.4810
Cash + Investments (bn),64.5090
Unlevered Beta,1.0378
Cash-Corrected Beta,1.0552


### Oracle

In [63]:
# Oracle's closing share price on 30 October 2026
orcl_price = yf.Ticker("ORCL").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

orcl_share_price = float(orcl_price["Close"].iloc[0])
print(f"Oracle share price: ${orcl_share_price:.2f}")

Oracle share price: $130.67


In [64]:
orcl_result = calculate_peer_beta(
    ticker="ORCL",
    share_price=orcl_share_price,
    shares_outstanding_bn=3.024,
    current_debt_bn=7.625,
    long_term_debt_bn=117.712,
    cash_bn=36.369,
    short_term_investments_bn=0.708
)

display(pd.DataFrame([orcl_result]).set_index("Ticker").T.round(4))

Ticker,ORCL
Equity Beta,1.7900
Market Cap (bn),395.1461
Debt (bn),125.3370
Cash + Investments (bn),37.0770
Unlevered Beta,1.3589
Cash-Corrected Beta,1.4632


### Nutanix

In [65]:
# Nutanix's closing share price on 30 October 2026
ntnx_price = yf.Ticker("NTNX").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

ntnx_share_price = float(ntnx_price["Close"].iloc[0])
print(f"Nutanix share price: ${ntnx_share_price:.2f}")

Nutanix share price: $67.21


In [66]:
ntnx_result = calculate_peer_beta(
    ticker="NTNX",
    share_price=ntnx_share_price,
    shares_outstanding_bn=0.270829,
    current_debt_bn=0,
    long_term_debt_bn=1.348711,
    cash_bn=0.777308,
    short_term_investments_bn=1.584155
)

display(pd.DataFrame([ntnx_result]).set_index("Ticker").T.round(4))

Ticker,NTNX
Equity Beta,0.7215
Market Cap (bn),18.2024
Debt (bn),1.3487
Cash + Investments (bn),2.3615
Unlevered Beta,0.6718
Cash-Corrected Beta,0.7640


### OVHcloud

In [67]:
# OVHcloud's closing share price on 30 October 2026
ovh_price = yf.Ticker("OVH.PA").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

ovh_share_price = float(ovh_price["Close"].iloc[0])
print(f"OVHcloud share price: €{ovh_share_price:.2f}")

OVHcloud share price: €13.25


In [68]:
ovh_result = calculate_peer_beta(
    ticker="OVH.PA",
    share_price=ovh_share_price,
    shares_outstanding_bn=0.153458722,
    current_debt_bn=0,
    long_term_debt_bn=1.2119,
    cash_bn=0.0553,
    short_term_investments_bn=0
)

display(pd.DataFrame([ovh_result]).set_index("Ticker").T.round(4))

Ticker,OVH.PA
Equity Beta,1.2615
Market Cap (bn),2.0333
Debt (bn),1.2119
Cash + Investments (bn),0.0553
Unlevered Beta,0.7904
Cash-Corrected Beta,0.8041


### AWS beta

In [69]:
# Combine all AWS peer results
aws_peers = pd.DataFrame([
    msft_result,
    orcl_result,
    ntnx_result,
    ovh_result
]).set_index("Ticker")

display(aws_peers.round(4))
aws_beta_table = aws_peers[
    ["Equity Beta", "Unlevered Beta", "Cash-Corrected Beta"]
].copy()

aws_beta_table.index = [
    "Microsoft",
    "Oracle",
    "Nutanix",
    "OVHcloud"
]

display(aws_beta_table.round(4))

,Equity Beta,Market Cap (bn),Debt (bn),Cash + Investments (bn),Unlevered Beta,Cash-Corrected Beta
Ticker,,,,,,
MSFT,1.0473,3876.2254,35.4810,64.5090,1.0378,1.0552
ORCL,1.7900,395.1461,125.3370,37.0770,1.3589,1.4632
NTNX,0.7215,18.2024,1.3487,2.3615,0.6718,0.7640
OVH.PA,1.2615,2.0333,1.2119,0.0553,0.7904,0.8041


,Equity Beta,Unlevered Beta,Cash-Corrected Beta
Microsoft,1.0473,1.0378,1.0552
Oracle,1.7900,1.3589,1.4632
Nutanix,0.7215,0.6718,0.7640
OVHcloud,1.2615,0.7904,0.8041


In [70]:
aws_beta = aws_peers["Cash-Corrected Beta"].mean()

print(f"AWS bottom-up operating beta: {aws_beta:.4f}")

AWS bottom-up operating beta: 1.0216


## North America

In [71]:
# North America peers and benchmark
tickers_na = [
    "WMT", "TGT", "EBAY", "ETSY", "SHOP",
    "^SP500TR"
]

# Download monthly adjusted prices
prices_na = yf.download(
    tickers_na,
    start="2021-10-01",
    end="2026-11-01",
    interval="1mo",
    auto_adjust=True,
    progress=False
)["Close"]

display(prices_na.tail())


Ticker,EBAY,ETSY,SHOP,TGT,WMT,^SP500TR
Date,,,,,,
2026-06-01,111.411263,75.330002,114.180000,128.708847,112.989502,16774.070312
2026-07-01,113.664413,81.680000,117.150002,142.386810,110.934418,16763.419922
2026-08-01,103.559998,81.330002,147.369995,159.755112,104.870003,17219.939453
2026-09-01,106.050003,71.489998,148.300003,155.594406,103.919998,17160.410156
2026-10-01,121.639999,88.959999,157.490005,153.300552,123.510002,17011.140625


In [72]:
# Calculate monthly returns

returns_na = prices_na.pct_change(fill_method=None).iloc[1:]

# Check the resulting dataset
print("Shape:", returns_na.shape)
print("\nMissing values:")
print(returns_na.isna().sum())

display(returns_na.head())

Shape: (60, 6)

Missing values:
Ticker
EBAY        0
ETSY        0
SHOP        0
TGT         0
WMT         0
^SP500TR    0
dtype: int64


Ticker,EBAY,ETSY,SHOP,TGT,WMT,^SP500TR
Date,,,,,,
2021-11-01,-0.118449,0.095297,0.037539,-0.057572,-0.058828,-0.006929
2021-12-01,-0.014231,-0.202637,-0.094888,-0.050853,0.033013,0.044816
2022-01-01,-0.096692,-0.282543,-0.299951,-0.047572,-0.033728,-0.051747
2022-02-01,-0.091227,-0.013942,-0.279993,-0.089798,-0.033259,-0.029942
2022-03-01,0.053256,-0.197624,-0.026359,0.062322,0.106066,0.037130


### OLS regression

In [73]:
# Run OLS regressions against S&P 500 Total Return Index
na_results = []

for ticker in ["WMT", "TGT", "EBAY", "ETSY"]:
    data = returns_na[[ticker, "^SP500TR"]].dropna()

    X = sm.add_constant(data["^SP500TR"])
    y = data[ticker]

    model = sm.OLS(y, X).fit()

    na_results.append({
        "Ticker": ticker,
        "Equity Beta": model.params["^SP500TR"],
        "Alpha": model.params["const"],
        "R-squared": model.rsquared,
        "Beta p-value": model.pvalues["^SP500TR"],
        "Observations": int(model.nobs)
    })

beta_results_na = pd.DataFrame(na_results).set_index("Ticker")
beta_results = pd.concat([beta_results, beta_results_na])

display(beta_results_na.round(4))


,Equity Beta,Alpha,R-squared,Beta p-value,Observations
Ticker,,,,,
WMT,0.5339,0.0126,0.1423,0.0030,60
TGT,0.9514,-0.0119,0.2213,0.0001,60
EBAY,1.2953,-0.0007,0.4283,0.0000,60
ETSY,1.7560,-0.0244,0.2592,0.0000,60


### WALMART

In [74]:
# Walmart's closing share price on 30 October 2026

wmt_price = yf.Ticker("WMT").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

wmt_share_price = float(wmt_price["Close"].iloc[0])
print(f"Walmart share price: ${wmt_share_price:.2f}")

Walmart share price: $123.51


In [75]:
wmt_result = calculate_peer_beta(
    ticker="WMT",
    share_price=wmt_share_price,
    shares_outstanding_bn=7.942,
    current_debt_bn=10.479 + 3.470,
    long_term_debt_bn=36.462,
    cash_bn=11.529,
    short_term_investments_bn=0
)

display(pd.DataFrame([wmt_result]).set_index("Ticker").T.round(4))

Ticker,WMT
Equity Beta,0.5339
Market Cap (bn),980.9164
Debt (bn),50.4110
Cash + Investments (bn),11.5290
Unlevered Beta,0.5078
Cash-Corrected Beta,0.5136


### TARGET

In [76]:
# Target's closing share price on 30 October 2026

tgt_price = yf.Ticker("TGT").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

tgt_share_price = float(tgt_price["Close"].iloc[0])
print(f"Target share price: ${tgt_share_price:.2f}")

Target share price: $154.38


In [77]:
tgt_result = calculate_peer_beta(
    ticker="TGT",
    share_price=tgt_share_price,
    shares_outstanding_bn=0.454291461,
    current_debt_bn=1.136,
    long_term_debt_bn=14.221,
    cash_bn=5.411,
    short_term_investments_bn=0
)

display(pd.DataFrame([tgt_result]).set_index("Ticker").T.round(4))

Ticker,TGT
Equity Beta,0.9514
Market Cap (bn),70.1335
Debt (bn),15.3570
Cash + Investments (bn),5.4110
Unlevered Beta,0.7805
Cash-Corrected Beta,0.8332


### EBAY

In [78]:
# eBays's closing share price on 30 October 2026

ebay_price = yf.Ticker("EBAY").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

ebay_share_price = float(ebay_price["Close"].iloc[0])
print(f"eBay share price: ${ebay_share_price:.2f}")

eBay share price: $121.64


In [79]:
ebay_result = calculate_peer_beta(
    ticker="EBAY",
    share_price=ebay_share_price,
    shares_outstanding_bn=0.439,
    current_debt_bn=1.599,
    long_term_debt_bn=5.134,
    cash_bn=1.166,
    short_term_investments_bn=0.809
)

display(pd.DataFrame([ebay_result]).set_index("Ticker").T.round(4))

Ticker,EBAY
Equity Beta,1.2953
Market Cap (bn),53.4000
Debt (bn),6.7330
Cash + Investments (bn),1.9750
Unlevered Beta,1.1503
Cash-Corrected Beta,1.1893


### ETSY

In [80]:
# Etsy's closing share price on 30 October 2026

etsy_price = yf.Ticker("ETSY").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

etsy_share_price = float(etsy_price["Close"].iloc[0])
print(f"Etsy share price: ${etsy_share_price:.2f}")

Etsy share price: $88.96


In [81]:
etsy_result = calculate_peer_beta(
    ticker="ETSY",
    share_price=etsy_share_price,
    shares_outstanding_bn=0.087914,
    current_debt_bn=1.298793,
    long_term_debt_bn=1.688345,
    cash_bn=2.065658,
    short_term_investments_bn=0.245043
)

display(pd.DataFrame([etsy_result]).set_index("Ticker").T.round(4))

Ticker,ETSY
Equity Beta,1.7560
Market Cap (bn),7.8208
Debt (bn),2.9871
Cash + Investments (bn),2.3107
Unlevered Beta,1.2707
Cash-Corrected Beta,1.6162


### North America beta

In [82]:
# Combine all North America peer results
na_peers = pd.DataFrame([
    wmt_result,
    tgt_result,
    ebay_result,
    etsy_result,
    #shop_result
]).set_index("Ticker")

display(na_peers.round(4))

na_beta_table = na_peers[
    ["Equity Beta", "Unlevered Beta", "Cash-Corrected Beta"]
].copy()

na_beta_table.index = [
    "Walmart",
    "Target",
    "eBay",
    "Etsy",
    #"Shopify"
]

display(na_beta_table.round(4))

,Equity Beta,Market Cap (bn),Debt (bn),Cash + Investments (bn),Unlevered Beta,Cash-Corrected Beta
Ticker,,,,,,
WMT,0.5339,980.9164,50.4110,11.5290,0.5078,0.5136
TGT,0.9514,70.1335,15.3570,5.4110,0.7805,0.8332
EBAY,1.2953,53.4000,6.7330,1.9750,1.1503,1.1893
ETSY,1.7560,7.8208,2.9871,2.3107,1.2707,1.6162


,Equity Beta,Unlevered Beta,Cash-Corrected Beta
Walmart,0.5339,0.5078,0.5136
Target,0.9514,0.7805,0.8332
eBay,1.2953,1.1503,1.1893
Etsy,1.7560,1.2707,1.6162


In [83]:
na_beta = na_peers["Cash-Corrected Beta"].mean()

print(f"North America bottom-up operating beta: {na_beta:.4f}")

North America bottom-up operating beta: 1.0381


## INTERNATIONAL

In [84]:
# International peers and benchmark
tickers_int = [
    "ZAL.DE", "ALE.WA", "CURY.L", "3092.T",
    "^SP500TR"
]

fx_tickers_int = [
    "EURUSD=X", "PLNUSD=X", "GBPUSD=X", "JPYUSD=X"
]

# Download monthly adjusted prices
prices_int = yf.download(
    tickers_int,
    start="2021-10-01",
    end="2026-11-01",
    interval="1mo",
    auto_adjust=True,
    progress=False
)["Close"]

# Download monthly exchange rates
fx_int = yf.download(
    fx_tickers_int,
    start="2021-10-01",
    end="2026-11-01",
    interval="1mo",
    auto_adjust=False,
    progress=False
)["Close"]

display(prices_int.tail())
display(fx_int.tail())

Ticker,3092.T,ALE.WA,CURY.L,ZAL.DE,^SP500TR
Date,,,,,
2026-06-01,1128.362427,37.814999,159.575867,25.370001,16774.070312
2026-07-01,1122.470093,44.985001,167.874603,28.260000,16763.419922
2026-08-01,1128.853394,45.610001,151.899994,24.650000,17219.939453
2026-09-01,1076.500000,50.630001,150.600006,22.700001,17160.410156
2026-10-01,1160.500000,59.549999,143.800003,26.719999,17011.140625


Ticker,EURUSD=X,GBPUSD=X,JPYUSD=X,PLNUSD=X
Date,,,,
2026-06-01,1.142204,1.325399,0.006176,0.266395
2026-07-01,1.152353,1.346094,0.006243,0.267691
2026-08-01,1.158896,1.354041,0.006245,0.267185
2026-09-01,1.134083,1.323276,0.006353,0.259828
2026-10-01,1.120699,1.339850,0.006346,0.253582


In [85]:
# Convert international share prices into USD
prices_usd_int = pd.DataFrame(index=prices_int.index)

prices_usd_int["ZAL.DE"] = (
    prices_int["ZAL.DE"] * fx_int["EURUSD=X"]
)

prices_usd_int["ALE.WA"] = (
    prices_int["ALE.WA"] * fx_int["PLNUSD=X"]
)

prices_usd_int["CURY.L"] = (
    prices_int["CURY.L"] / 100 * fx_int["GBPUSD=X"]
)

prices_usd_int["3092.T"] = (
    prices_int["3092.T"] * fx_int["JPYUSD=X"]
)

prices_usd_int["^SP500TR"] = prices_int["^SP500TR"]

display(prices_usd_int.tail())

,ZAL.DE,ALE.WA,CURY.L,3092.T,^SP500TR
Date,,,,,
2026-06-01,28.977727,10.073738,2.115016,6.968512,16774.070312
2026-07-01,32.565484,12.042081,2.259750,7.007423,16763.419922
2026-08-01,28.566791,12.186292,2.056789,7.049958,17219.939453
2026-09-01,25.743675,13.155092,1.992854,6.839089,17160.410156
2026-10-01,29.945084,15.100799,1.926705,7.364747,17011.140625


In [86]:
# Calculate monthly returns
returns_int = prices_usd_int.pct_change(fill_method=None).iloc[1:]

display(returns_int.head())
print("Number of observations:", len(returns_int))
print("\nMissing values:")
print(returns_int.isna().sum())

,ZAL.DE,ALE.WA,CURY.L,3092.T,^SP500TR
Date,,,,,
2021-11-01,-0.038980,-0.160974,0.046318,-0.007960,-0.006929
2021-12-01,-0.111151,0.009199,-0.111906,-0.017782,0.044816
2022-01-01,-0.031303,-0.045000,-0.077343,-0.157446,-0.051747
2022-02-01,-0.146424,-0.184816,-0.118636,0.069610,-0.029942
2022-03-01,-0.229256,0.146753,-0.048916,-0.031869,0.037130


Number of observations: 60

Missing values:
ZAL.DE      0
ALE.WA      0
CURY.L      0
3092.T      0
^SP500TR    0
dtype: int64


### OLS regression

In [87]:
market_returns_int = returns_int["^SP500TR"]

results_int = []

for ticker_int in ["ZAL.DE", "ALE.WA", "CURY.L", "3092.T"]:

    # Dependent variable: peer stock returns
    y_int = returns_int[ticker_int]

    # Independent variable: market returns + intercept
    X_int = sm.add_constant(market_returns_int)

    # Estimate OLS regression
    model_int = sm.OLS(y_int, X_int).fit()

    results_int.append({
        "Ticker": ticker_int,
        "Equity Beta": model_int.params["^SP500TR"],
        "Alpha": model_int.params["const"],
        "R-squared": model_int.rsquared,
        "Beta p-value": model_int.pvalues["^SP500TR"],
        "Observations": int(model_int.nobs)
    })

beta_results_int = pd.DataFrame(results_int).set_index("Ticker")
beta_results = pd.concat([beta_results, beta_results_int])

display(beta_results_int.round(4))

,Equity Beta,Alpha,R-squared,Beta p-value,Observations
Ticker,,,,,
ZAL.DE,1.4199,-0.0232,0.1873,0.0006,60
ALE.WA,1.0842,0.0001,0.1828,0.0007,60
CURY.L,1.3101,-0.0017,0.1720,0.0010,60
3092.T,0.9981,-0.0108,0.2694,0.0000,60


### ZALANDO

In [88]:
# Zalando's closing share price on 30 October 2026

zal_price = yf.Ticker("ZAL.DE").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

zal_share_price = float(zal_price["Close"].iloc[0])
print(f"Zalando share price: €{zal_share_price:.2f}")

Zalando share price: €26.72


In [89]:
zal_result = calculate_peer_beta(
    ticker="ZAL.DE",
    share_price=zal_share_price,
    shares_outstanding_bn=0.2460,
    current_debt_bn=0,
    long_term_debt_bn=0.4870,
    cash_bn=1.3978 - 0.1755,    # exclude third-party escrow funds
    short_term_investments_bn=0
)

display(pd.DataFrame([zal_result]).set_index("Ticker").T.round(4))

Ticker,ZAL.DE
Equity Beta,1.4199
Market Cap (bn),6.5731
Debt (bn),0.4870
Cash + Investments (bn),1.2223
Unlevered Beta,1.3220
Cash-Corrected Beta,1.5988


### ALLEGRO

In [90]:
# Allegro's closing share price on 30 October 2026

ale_price = yf.Ticker("ALE.WA").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

ale_share_price = float(ale_price["Close"].iloc[0])
print(f"Allegro share price: PLN {ale_share_price:.2f}")

Allegro share price: PLN 59.55


In [91]:
ale_result = calculate_peer_beta(
    ticker="ALE.WA",
    share_price=ale_share_price,
    shares_outstanding_bn=1.000369558,
    current_debt_bn=0,
    long_term_debt_bn=4.9744,
    cash_bn=3.0448,
    short_term_investments_bn=0
)

display(pd.DataFrame([ale_result]).set_index("Ticker").T.round(4))

Ticker,ALE.WA
Equity Beta,1.0842
Market Cap (bn),59.5720
Debt (bn),4.9744
Cash + Investments (bn),3.0448
Unlevered Beta,1.0006
Cash-Corrected Beta,1.0502


### ZOZO

In [92]:
# Zozo's closing share price on 30 October 2026

zozo_price = yf.Ticker("3092.T").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

zozo_share_price = float(zozo_price["Close"].iloc[0])
print(f"ZOZO share price: ¥{zozo_share_price:.2f}")

ZOZO share price: ¥1160.50


In [93]:
zozo_result = calculate_peer_beta(
    ticker="3092.T",
    share_price=zozo_share_price,
    shares_outstanding_bn=0.868514726,
    current_debt_bn=20.001,
    long_term_debt_bn=0,
    cash_bn=33.291,
    short_term_investments_bn=0
)

display(pd.DataFrame([zozo_result]).set_index("Ticker").T.round(4))

Ticker,3092.T
Equity Beta,0.9981
Market Cap (bn),1007.9113
Debt (bn),20.0010
Cash + Investments (bn),33.2910
Unlevered Beta,0.9787
Cash-Corrected Beta,1.0115


### CURRYS

In [94]:
# Currys' closing share price on 30 October 2026

cur_price = yf.Ticker("CURY.L").history(
    start="2026-10-30",
    end="2026-10-31",
    auto_adjust=False
)

cur_share_price = float(cur_price["Close"].iloc[0]) / 100
print(f"Currys share price: £{cur_share_price:.4f}")

Currys share price: £1.4380


In [95]:
cur_result = calculate_peer_beta(
    ticker="CURY.L",
    share_price=cur_share_price,
    shares_outstanding_bn=1.077823361,
    current_debt_bn=0,
    long_term_debt_bn=0,
    cash_bn=0.176,
    short_term_investments_bn=0
)

display(pd.DataFrame([cur_result]).set_index("Ticker").T.round(4))

Ticker,CURY.L
Equity Beta,1.3101
Market Cap (bn),1.5499
Debt (bn),0.0000
Cash + Investments (bn),0.1760
Unlevered Beta,1.3101
Cash-Corrected Beta,1.4779


### International beta

In [96]:
# Combine all International peer results
int_peers = pd.DataFrame([
    zal_result,
    ale_result,
    cur_result,
    zozo_result
]).set_index("Ticker")

display(int_peers.round(4))

int_beta_table = int_peers[
    ["Equity Beta", "Unlevered Beta", "Cash-Corrected Beta"]
].copy()

int_beta_table.index = [
    "Zalando",
    "Allegro",
    "Currys",
    "ZOZO"
]

display(int_beta_table.round(4))

,Equity Beta,Market Cap (bn),Debt (bn),Cash + Investments (bn),Unlevered Beta,Cash-Corrected Beta
Ticker,,,,,,
ZAL.DE,1.4199,6.5731,0.4870,1.2223,1.3220,1.5988
ALE.WA,1.0842,59.5720,4.9744,3.0448,1.0006,1.0502
CURY.L,1.3101,1.5499,0.0000,0.1760,1.3101,1.4779
3092.T,0.9981,1007.9113,20.0010,33.2910,0.9787,1.0115


,Equity Beta,Unlevered Beta,Cash-Corrected Beta
Zalando,1.4199,1.3220,1.5988
Allegro,1.0842,1.0006,1.0502
Currys,1.3101,1.3101,1.4779
ZOZO,0.9981,0.9787,1.0115


In [97]:
int_beta = int_peers["Cash-Corrected Beta"].mean()

print(f"International bottom-up operating beta: {int_beta:.4f}")

International bottom-up operating beta: 1.2846


## Amazon's beta

In [98]:
# Amazon segment data (9M 2026, USD millions)
amazon_operating_income = pd.Series({
    "AWS": 48555,
    "North America": 26113,
    "International": 4721
})

amazon_revenue = pd.Series({
    "AWS": 125803,
    "North America": 333172,
    "International": 125094
})

# Segment bottom-up operating betas
amazon_segment_betas = pd.Series({
    "AWS": aws_beta,
    "North America": na_beta,
    "International": int_beta
})

# Calculate segment weights
amazon_weights = amazon_operating_income / amazon_operating_income.sum()
amazon_revenue_weights = amazon_revenue / amazon_revenue.sum()

# Calculate weighted operating betas
amazon_operating_beta = (
    amazon_weights * amazon_segment_betas
).sum()

amazon_operating_beta_revenue = (
    amazon_revenue_weights * amazon_segment_betas
).sum()

# Display comparison
display(pd.DataFrame({
    "Operating Income ($m)": amazon_operating_income,
    "OI Weight": amazon_weights,
    "Revenue ($m)": amazon_revenue,
    "Revenue Weight": amazon_revenue_weights,
    "Operating Beta": amazon_segment_betas
}).round(4))

print(f"Operating income-weighted beta: {amazon_operating_beta:.4f}")
print(f"Revenue-weighted beta: {amazon_operating_beta_revenue:.4f}")

# Display segment operating betas
segment_beta_table = pd.DataFrame({
    "Operating Beta": amazon_segment_betas
})

display(segment_beta_table.round(4))

,Operating Income ($m),OI Weight,Revenue ($m),Revenue Weight,Operating Beta
AWS,48555,0.6116,125803,0.2154,1.0216
North America,26113,0.3289,333172,0.5704,1.0381
International,4721,0.0595,125094,0.2142,1.2846


Operating income-weighted beta: 1.0427
Revenue-weighted beta: 1.0873


,Operating Beta
AWS,1.0216
North America,1.0381
International,1.2846


In [99]:
# Amazon capital structure (USD billions)

amzn_share_price = 240.35
amzn_shares_outstanding_bn = 10.811

amzn_market_cap = amzn_share_price * amzn_shares_outstanding_bn

amzn_current_debt = 6.827
amzn_long_term_debt = 173.125
amzn_total_debt = amzn_current_debt + amzn_long_term_debt

amzn_cash = 55.783
amzn_marketable_securities = 58.443
amzn_total_cash = amzn_cash + amzn_marketable_securities

# Convert operating beta into equity beta

amzn_relevering_factor = (
    amzn_market_cap + amzn_total_debt - amzn_total_cash
) / amzn_market_cap

amzn_equity_beta_income = amazon_operating_beta * amzn_relevering_factor
amzn_equity_beta_revenue = amazon_operating_beta_revenue * amzn_relevering_factor

print(f"Amazon market cap: ${amzn_market_cap:.2f}bn")
print(f"Amazon total debt: ${amzn_total_debt:.2f}bn")
print(f"Amazon cash + securities: ${amzn_total_cash:.2f}bn")
print(f"Relevering factor: {amzn_relevering_factor:.4f}")
print(f"Bottom-up equity beta (income): {amzn_equity_beta_income:.4f}")
print(f"Bottom-up equity beta (revenue): {amzn_equity_beta_revenue:.4f}")

Amazon market cap: $2598.42bn
Amazon total debt: $179.95bn
Amazon cash + securities: $114.23bn
Relevering factor: 1.0253
Bottom-up equity beta (income): 1.0690
Bottom-up equity beta (revenue): 1.1148


In [100]:
# CAPM inputs
rf = 0.0529
erp = 0.0441

# Bottom-up cost of equity
ke_income = rf + amzn_equity_beta_income * erp
ke_revenue = rf + amzn_equity_beta_revenue * erp

# Top-down cost of equity
ke_topdown = rf + 1.4917 * erp

print(f"Bottom-up cost of equity (income): {ke_income:.2%}")
print(f"Bottom-up cost of equity (revenue): {ke_revenue:.2%}")
print(f"Top-down cost of equity: {ke_topdown:.2%}")

Bottom-up cost of equity (income): 10.00%
Bottom-up cost of equity (revenue): 10.21%
Top-down cost of equity: 11.87%


In [101]:
# Summary of Amazon's bottom-up beta estimates
bottom_up_summary = pd.DataFrame({
    "Operating Income-Weighted": [
        amazon_operating_beta,
        amzn_equity_beta_income,
        ke_income
    ],
    "Revenue-Weighted": [
        amazon_operating_beta_revenue,
        amzn_equity_beta_revenue,
        ke_revenue
    ]
}, index=[
    "Operating Beta",
    "Equity Beta",
    "Cost of Equity"
])

# Format results
bottom_up_summary_formatted = bottom_up_summary.astype(str)

for col in bottom_up_summary_formatted.columns:
    bottom_up_summary_formatted.loc["Operating Beta", col] = (
        f"{bottom_up_summary.loc['Operating Beta', col]:.4f}"
    )
    bottom_up_summary_formatted.loc["Equity Beta", col] = (
        f"{bottom_up_summary.loc['Equity Beta', col]:.4f}"
    )
    bottom_up_summary_formatted.loc["Cost of Equity", col] = (
        f"{bottom_up_summary.loc['Cost of Equity', col]:.2%}"
    )

display(bottom_up_summary_formatted)


,Operating Income-Weighted,Revenue-Weighted
Operating Beta,1.0427,1.0873
Equity Beta,1.0690,1.1148
Cost of Equity,10.00%,10.21%


In [102]:
# Final comparison: Bottom-Up vs Top-Down
final_comparison = pd.DataFrame({
    "Bottom-Up": [
        f"{amzn_equity_beta_income:.4f}",
        f"{ke_income:.2%}"
    ],
    "Top-Down": [
        "1.4917",
        "11.87%"
    ]
}, index=[
    "Equity Beta",
    "Cost of Equity"
])

display(final_comparison)

,Bottom-Up,Top-Down
Equity Beta,1.0690,1.4917
Cost of Equity,10.00%,11.87%
